# Step 0 — Shared Imports, Hardware Setup, and Helper Functions

**Purpose:** Reference notebook. Every other notebook (`01` through `07`) is self-contained and copies these cells.

Run this to verify your environment is working before moving to any circuit notebook.

In [ ]:
import os
import dataclasses
import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.backends import TrajectoryBackend, ManyShotRunner
from noisiq.suppression import apply_dd
from noisiq.noise import fill_idle_with_identities
from noisiq.visualization import (
    Visualizer,
    export_gif,
    interactive_heatmap,
    plot_error_heatmap,
    per_qubit_purities,
    add_purity_panel,
)

os.makedirs("outputs", exist_ok=True)

profile    = nq.noise.get_hardware("ibm_eagle_r3")
gate_times = profile.gate_times

# DD demo profile — adds quasi-static Z component for DD to refocus
demo_profile_dd = dataclasses.replace(profile, idle_coherent_epsilon=0.07)

N_SHOTS      = 2000   # ManyShotRunner (heatmap / GIF Pauli pass)
N_SHOTS_TRAJ = 400    # TrajectoryBackend (density matrix → purity)

print(f"noisiq {nq.__version__}")
print(profile.describe())

In [ ]:
def controlled_phase(c: nq.Circuit, ctrl: int, tgt: int, theta: float, t=None) -> None:
    """Decompose ctrl-P(θ) into 4 standard gates."""
    c.p(ctrl, theta / 2, t=t)
    c.cnot(ctrl, tgt)
    c.p(tgt, -theta / 2)
    c.cnot(ctrl, tgt)
    c.p(tgt,  theta / 2)


def run_purity_pass(circuit, noise_config_twirl, n_qubits, label=""):
    """Run TrajectoryBackend and print per-qubit purities."""
    result_traj = TrajectoryBackend().run(
        circuit,
        noise_model=noise_config_twirl,
        n_shots=N_SHOTS_TRAJ,
        seed=42,
    )
    rho = result_traj.final_state
    purities = per_qubit_purities(rho, n_qubits)

    print(f"\n{'─'*50}")
    print(f"Per-qubit purity  [{label}]")
    for q, p in enumerate(purities):
        bar = "█" * int(p * 20)
        print(f"  q{q:>2d}  Tr(ρ²) = {p:.4f}  {bar}")
    print(f"{'─'*50}\n")
    return rho, purities


def visualize_circuit(circuit, result_many, noise_pauli, rho, purities, label: str, gif_name: str) -> None:
    """Static heatmap + interactive heatmap + GIF export."""
    fig = plot_error_heatmap(
        result_many, circuit,
        noise_config=noise_pauli,
        title=f"IBM Eagle r3 · {label}",
    )
    add_purity_panel(fig, fig.axes[0], rho, circuit.n_qubits)
    plt.show()

    interactive_heatmap(
        result_many, circuit,
        noise_config=noise_pauli,
        display_mode="annotate",
        title=f"IBM Eagle r3 · {label} [interactive]",
    )
    plt.show()

    viz = Visualizer(circuit)
    viz.run_single(noise_config=noise_pauli, seed=7)
    gif_path = f"outputs/{gif_name}.gif"
    export_gif(viz, gif_path, purities=purities)
    print(f"GIF → {gif_path}")


print("Helpers defined.")